# Suite TRF — Transforms

A transform is a rewrite whose `before` and `after` are predicates over the same symbols, with parameters typed by value
domains. A session keeps the candidates: steps enabled where `before` holds and `after` does not, one per value of each
finite domain, ordered with parameter decisions first. It takes one step per decision, as the caller decides or, stepping
over, as a policy ranks; a composite transform is stepped into or over. The steps taken are a trace, written as data.

In [ ]:
from mbse.Expressions import Expressions as E
from mbse.Patterns import Predicates as P, Queries as Q, Transforms as T
from mbse.Schemas.Framework import JSON, Proxies, Reflection as R, Schemas as S, Validators as SV
from support import native, raises

Held = S.OfRelation.Builder().name("Held").links("shelf", "item").create()
Shelf = S.OfObject.Builder().name("Shelf").ref().singleton("Shelf").relations(lambda r: r.name("items").of(Held).me("shelf")).create()
Item = S.OfObject.Builder().name("Item").ref().properties(native("name"), native("label"), native("size", int)).relations(
    lambda r: r.name("shelves").of(Held).me("item")).create()
i = E.variable("i")


def over(constraint=None):
    """A predicate over one item, `i`."""
    builder = P.OfPredicate.Builder().symbols({"i": Item})
    return (builder if constraint is None else builder.requires(constraint)).create()


def shelf(*names):
    """A store whose shelf holds an item of each name, in order (None for an item without a name)."""
    store = Proxies.OfStore()
    for schema in (Shelf, Item, Held):
        store.register(schema)
    for name in names:
        item = store.Item().create() if name is None else store.Item().name(name).create()
        store.Shelf(store.singleton("Shelf")).items(lambda e, x=item: e.item(x)).update()
    return store


Option = S.OfObject.Builder().create()
Case = S.OfUnion.Builder().branches(lambda b: b.name("upper").of(Option), lambda b: b.name("lower").of(Option)).create()


def label(store, match, arguments):
    item = match["i"]
    store.Item(item).label(item.name.upper() if arguments["case"] == "upper" else item.name.lower()).update()


def size(store, match, arguments):
    store.Item(match["i"]).size(arguments["size"]).update()


Label = T.Transform("Label", over(i.has("name")), over(i.has("label")), [lambda p: p.name("case").of(Case)], label)
Size = T.Transform("Size", over(i.has("label")), over(i.has("size")), [lambda p: p.name("size").of(lambda t: t.as_native(int))], size)
Tag = T.Transform("Tag", over(i.has("name")), over(i.has("label")),
                  rewrite=lambda store, match, arguments: store.Item(match["i"]).label("tag").update())
Finish = T.Transform("Finish", over(i.has("name")), over(i.has("label").and_(i.has("size"))), parts=[Label, Size])


def seen(candidates):
    """Each candidate as its transform, item and arguments."""
    return [(c.transform.name, c.match["i"].name, dict(c.arguments)) for c in candidates]

## TRF-01 · A transform: before and after over the same symbols, parameters typed by domains, a rewrite or parts

In [ ]:
assert (Label.name, list(Label.parameters), Label.parameters["case"].type is Case, Label.parts) == ("Label", ["case"], True, ())
assert Label.check() == [] and Finish.check() == [] and Finish.rewrite is None and Finish.parts == (Label, Size)
assert list(T.Transform("X", over(), over(), [S.OfParameter.Data("n")], size).parameters) == ["n"]  # data or specs
Other = S.OfObject.Builder().name("Other").ref().create()
odd = T.Transform("Odd", over(), P.OfPredicate.Builder().symbols({"j": Item}).create(), rewrite=size)
assert odd.check() == ["transform 'Odd': before and after must have the same symbols"]
assert T.Transform("Bare", over(), over()).check() == ["transform 'Bare': a transform has a rewrite or parts, one of them"]
assert T.Transform("Both", over(), over(), rewrite=size, parts=[Label]).check() == [
    "transform 'Both': a transform has a rewrite or parts, one of them"]
assert T.Transform("Outer", over(), over(), parts=[odd]).check() == ["transform 'Odd': before and after must have the same symbols"]
with raises(ValueError, match="transform 'Odd': before and after must have the same symbols; transform 'Bare': a transform"):
    T.Session(shelf(), [odd, T.Transform("Bare", over(), over())])

## TRF-02 · Candidates: enabled where before holds and after does not, finite domains expanded, parameter decisions first

In [ ]:
store = shelf("bolt", None, "nut")
Big = T.Transform("Big", over(i.get("size").gt(2)), over(i.has("label")), rewrite=size)
session = T.Session(store, [Tag, Label, Size, Big])
assert seen(session.candidates()) == [  # Label's, which decide a parameter, first; an item without a name matches neither
    ("Label", "bolt", {"case": "upper"}), ("Label", "bolt", {"case": "lower"}),
    ("Label", "nut", {"case": "upper"}), ("Label", "nut", {"case": "lower"}),
    ("Tag", "bolt", {}), ("Tag", "nut", {})]
assert session.undecided == ["'Big' at i=Item#0: its before is unknown", "'Big' at i=Item#1: its before is unknown",
                             "'Big' at i=Item#2: its before is unknown"]  # no size: whether it exceeds 2 is unknown
everything = T.Session(shelf("bolt"), [T.Transform("All", over(True), over(i.get("size").gt(2)), rewrite=size)])
assert everything.candidates() == [] and everything.undecided == ["'All' at i=Item#0: its after is unknown"]  # before: anything
first = session.candidates()[0]
assert first.open == () and first.score == 0.0 and first.match["i"].name == "bolt"
store.Item(first.match["i"]).label("BOLT").update()  # outside the session: found again only after a step
loose = S.OfUnion.Builder().branches(lambda b: b.name("a").of(Item), lambda b: b.name("b").of(Option)).create()
Free = T.Transform("Free", over(i.has("name")), over(i.has("size")), [lambda p: p.name("n").of(loose), lambda p: p.name("on").of(
    lambda t: t.as_native(bool))], size)
free = T.Session(shelf("bolt"), [Free]).candidates()
assert seen(free) == [("Free", "bolt", {"on": False}), ("Free", "bolt", {"on": True})] and free[0].open == ("n",)  # not options
answered = free[0].answer(n=3)
assert answered.open == () and answered.arguments == {"on": False, "n": 3} and free[0].open == ("n",)
with raises(TypeError, match="'Free' has no open parameter 'on'"):
    free[0].answer(on=True)

## TRF-03 · Taking a step: one decision, resolved linearly; refusals; a rewrite must establish its after

In [ ]:
store = shelf("bolt", "nut")
session = T.Session(store, [Label, Size])
bolt_upper = session.candidates()[0]
session.take(bolt_upper)
assert store.extent("Item")[0].name == "nut"  # the rewrite moved bolt last in its extent; labels keep their first order
assert seen(session.candidates()) == [("Label", "nut", {"case": "upper"}), ("Label", "nut", {"case": "lower"}),
                                      ("Size", "bolt", {})]  # bolt's other case is no longer enabled
with raises(ValueError, match="'Label' is not enabled for that match"):
    session.take(bolt_upper)
size_bolt = session.candidates()[2]
with raises(ValueError, match="'Size' has open parameters ['size']"):
    session.take(size_bolt)
session.take(size_bolt.answer(size=3))
assert (bolt_upper.match["i"].label, bolt_upper.match["i"].size) == ("BOLT", 3)
with raises(TypeError, match="'Label' is not composite: take it"):
    session.step_in(session.candidates()[0])
with raises(TypeError, match="'Finish' is composite: step in or over it"):
    T.Session(shelf("bolt"), [Finish]).take(T.Session(shelf("bolt"), [Finish]).candidates()[0])
broken = T.Transform("Broken", over(i.has("name")), over(i.has("label")), rewrite=lambda store, match, arguments: None)
lazy = T.Session(shelf("bolt"), [broken])
with raises(ValueError, match="'Broken' did not establish its after at i=Item#0"):
    lazy.take(lazy.candidates()[0])

## TRF-04 · A policy ranks candidates: a recommendation, or the decision where the caller steps over

In [ ]:
store = shelf("bolt", "nut")
session = T.Session(store, [Label, Tag, Size])
policy = T.Policy(T.Clause("Label", {"case": "lower"}, 2.0), T.Clause("Tag"), T.Clause("Size", {"size": 5}))
ranked = session.candidates(policy)
assert [(c.transform.name, c.match["i"].name, dict(c.arguments), c.score) for c in ranked] == [
    ("Label", "bolt", {"case": "lower"}, 2.0), ("Label", "nut", {"case": "lower"}, 2.0),
    ("Tag", "bolt", {}, 1.0), ("Tag", "nut", {}, 1.0),
    ("Label", "bolt", {"case": "upper"}, 0.0), ("Label", "nut", {"case": "upper"}, 0.0)]  # by score, then in order
assert session.step_over(policy) is True and store.extent("Item")[1].label == "bolt"  # the policy's first
assert session.run(policy) == 3 and session.done  # nut's label, then each size, which the clause answers
assert [(item.label, item.size) for item in store.extent("Item")] == [("bolt", 5), ("nut", 5)]
assert [step.by for step in session.steps] == ["policy"] * 4
cautious = T.Session(shelf("bolt"), [Label, Size])
assert cautious.step_over(T.Policy(T.Clause("Size"))) is False  # no clause weighs Label: not the policy's to take
cautious.take(cautious.candidates()[0])
assert cautious.step_over(T.Policy(T.Clause("Size"))) is False and not cautious.done  # it cannot answer size
assert cautious.run(T.Policy(T.Clause("Size", {"size": 1}))) == 1 and cautious.done and cautious.step_over(policy) is False
assert T.Policy(T.Clause("Free", {"on": True, "n": 7})).answers(free[1]) == {"n": 7}  # only open parameters are answered

## TRF-05 · Composite transforms: stepped into, the caller decides inside; stepped over, the policy does

In [ ]:
store = shelf("bolt", "nut")
session = T.Session(store, [Finish])
assert seen(session.candidates()) == [("Finish", "bolt", {}), ("Finish", "nut", {})]
inner = session.step_in(session.candidates()[0])
assert seen(inner.candidates()) == [("Label", "bolt", {"case": "upper"}), ("Label", "bolt", {"case": "lower"})]  # its match only
with raises(ValueError, match="'Finish' is open: take its steps first"):
    session.candidates()
inner.take(inner.candidates()[1])
inner.take(inner.candidates()[0].answer(size=2))
assert inner.done and seen(session.candidates()) == [("Finish", "nut", {})]  # recorded once its session is done
assert session.step_over(T.Policy(T.Clause("Finish"), T.Clause("Label", {"case": "upper"}), T.Clause("Size", {"size": 4})))
assert session.done and [(s.transform, s.by, s.mode, [(x.transform, x.by) for x in s.steps]) for s in session.steps] == [
    ("Finish", "caller", "in", [("Label", "caller"), ("Size", "caller")]),
    ("Finish", "policy", "over", [("Label", "policy"), ("Size", "policy")])]
Wrap = T.Transform("Wrap", over(i.has("name")), over(i.has("label")), [lambda p: p.name("tag")], parts=[Label])
wrapped = T.Session(shelf("bolt"), [Wrap])
with raises(ValueError, match="'Wrap' has open parameters ['tag']"):
    wrapped.step_in(wrapped.candidates()[0])
partial = T.Session(shelf("bolt"), [Finish])
assert partial.step_over(T.Policy(T.Clause("Finish"), T.Clause("Label", {"case": "lower"}))) is True  # opened, then stuck
assert not partial.done and partial.step_over(T.Policy(T.Clause("Size", {"size": 1}))) is True  # steps inside
assert partial.done and partial.steps[0].mode == "over" and [x.transform for x in partial.steps[0].steps] == ["Label", "Size"]

## TRF-06 · The trace is data: written byte-identically, read back, validated; two runs write the same

In [ ]:
def finished(names):
    store = shelf(*names)
    session = T.Session(store, [Finish, Tag])
    session.step_in(session.candidates()[0])
    session.run(T.Policy(T.Clause("Finish"), T.Clause("Label", {"case": "upper"}), T.Clause("Size", {"size": 2})))
    return session


traces = T.register(Proxies.OfStore())
assert T.register(traces) is traces  # once is enough
session = finished(["bolt", "nut"])
text = JSON.ToJSON(traces)(T.Trace, session.trace(traces))
assert text == JSON.ToJSON(traces)(T.Trace, finished(["bolt", "nut"]).trace(traces))  # deterministic
assert text == (
    '{"root": "s0", "objects": {"s0": {"steps": ['
    '{"transform": "Finish", "match": [{"symbol": "i", "element": "Item#0"}], "by": "caller", "mode": "in", "steps": ['
    '{"transform": "Label", "match": [{"symbol": "i", "element": "Item#0"}], "arguments": [{"name": "case", "value": {"str": "upper"}}], "by": "policy"}, '
    '{"transform": "Size", "match": [{"symbol": "i", "element": "Item#0"}], "arguments": [{"name": "size", "value": {"int": 2}}], "by": "policy"}]}, '
    '{"transform": "Finish", "match": [{"symbol": "i", "element": "Item#1"}], "by": "policy", "mode": "over", "steps": ['
    '{"transform": "Label", "match": [{"symbol": "i", "element": "Item#1"}], "arguments": [{"name": "case", "value": {"str": "upper"}}], "by": "policy"}, '
    '{"transform": "Size", "match": [{"symbol": "i", "element": "Item#1"}], "arguments": [{"name": "size", "value": {"int": 2}}], "by": "policy"}]}]}}}')
back = JSON.FromJSON(traces)(T.Trace, text)
assert JSON.ToJSON(traces)(T.Trace, back) == text and SV.Validate(traces)(T.Trace, back) == []
Note = T.Transform("Note", over(i.has("name")), over(i.has("label")), [lambda p: p.name("note")],
                   lambda store, match, arguments: store.Item(match["i"]).label("noted").update())
noted = T.Session(shelf("bolt"), [Note])
noted.take(noted.candidates()[0].answer(note=None))
with raises(TypeError, match="argument 'note' is not a native value: None"):
    noted.trace(traces)

## TRF-07 · Transforms match schemas, each an object of its kind's meta-schema (mbse-schemas' `Reflection`)

In [ ]:
Phone = S.OfObject.Builder().name("Phone").properties(native("number")).create()
Contact = S.OfObject.Builder().name("Contact").ref().properties(native("name"), native("number"),
                                                               lambda p: p.name("home").of(Phone)).create()
source = Proxies.OfStore()
source.register(Contact)
s, t, p = E.variable("s"), E.variable("t"), E.variable("p")
named = lambda constraint: P.OfPredicate.Builder().symbols({"s": S.OfObject.Schema}).requires(constraint).create()  # noqa: E731
Document = T.Transform("Document", named(s.has("name")), named(s.has("description")),
                       rewrite=lambda store, match, arguments: setattr(match["s"], "description", f"A {match['s'].name}"))
schemas = R.of(source)  # what the store registers, and Phone, which Contact refers to
documenting = T.Session(schemas, [Document])
assert [c.match["s"] for c in documenting.candidates()] == [Contact, Phone]  # the schemas themselves
assert documenting.run(T.Policy(T.Clause("Document"))) == 2 and documenting.done
assert (Contact.description, Phone.description) == ("A Contact", "A Phone")
assert [step.match for step in documenting.steps] == [(("s", "Schemas.Object#0"),), (("s", "Schemas.Object#1"),)]
# Within a schema, values: read with get, quantified over and compared deeply.
Shares = P.OfPredicate.Builder().name("Shares").symbols({"s": S.OfObject.Schema, "t": S.OfObject.Schema}).requires(
    s.ne(t).and_(s.get("properties").any("p", p.in_(t.get("properties"))))).create()
assert [(m["s"].name, m["t"].name) for m in Q.select(schemas, Shares)] == [("Contact", "Phone"), ("Phone", "Contact")]  # number